# PatchTST + VMD v4 -- Turkiye Elektrik Yuk Tahmini

## v4 Degisiklikleri
- **41 il** hava verisi -- tüketim oranıyla ağırlıklandırılmış
- **2010-2026** arası veri (16 yıl)
- **Ağırlıklı hava**: İstanbul %24.43, İzmir %11.51, vs.
- **Scaler leakage yok**: sadece train ile fit
- **Explicit feature list**
- **Global quantile PeakAwareLoss**

## Kaggle Kurulum
1. Sol menu 'Add Input' -> epias CSV dosyasini yukle
2. Session options -> GPU P100 sec
3. Internet acik olmali

In [ ]:
# ============================================================
# HUCRE 1 -- GPU KONTROL
# ============================================================
import torch
print('PyTorch: ' + torch.__version__)
print('CUDA: ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    print('GPU: ' + torch.cuda.get_device_name(0))
    print('Bellek: {:.1f} GB'.format(torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz: ' + str(DEVICE))

In [ ]:
# ============================================================
# HUCRE 2 -- KURULUM
# ============================================================
!pip install vmdpy holidays pvlib -q
print('Kurulum tamamlandi.')

In [ ]:
# ============================================================
# HUCRE 3 -- KAGGLE DOSYA YOLU
# ============================================================
import os

OUTPUT_DIR = '/kaggle/working/'
EPIAS_PATH = ''

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv') and ('epias' in f.lower() or 'tuketim' in f.lower() or 'load' in f.lower() or 'consumption' in f.lower()):
            EPIAS_PATH = os.path.join(root, f)
            print('EPİAS bulundu: ' + EPIAS_PATH)
            break

if not EPIAS_PATH:
    print('UYARI: EPİAS bulunamadi, demo veri kullanilacak.')

In [ ]:
# ============================================================
# HUCRE 4 -- KUTUPHANELER + KONFIGURASYON v4
# ============================================================
import requests
import pandas as pd
import numpy as np
import time, datetime, warnings, os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from vmdpy import VMD
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from tqdm.notebook import tqdm

# ---- 41 İL: koordinat + tuketim agirligi ----
SEHIRLER = {
    'afyonkarahisar': {'lat': 38.76, 'lon': 30.54, 'oran': 1.52},
    'agri':           {'lat': 39.72, 'lon': 43.05, 'oran': 0.00},
    'antalya':        {'lat': 36.90, 'lon': 30.70, 'oran': 8.20},
    'aydin':          {'lat': 37.85, 'lon': 27.84, 'oran': 2.73},
    'balikesir':      {'lat': 39.65, 'lon': 27.89, 'oran': 2.00},
    'bilecik':        {'lat': 40.15, 'lon': 29.98, 'oran': 0.77},
    'bitlis':         {'lat': 38.40, 'lon': 42.12, 'oran': 0.43},
    'burdur':         {'lat': 37.72, 'lon': 30.29, 'oran': 0.52},
    'bursa':          {'lat': 40.18, 'lon': 29.06, 'oran': 4.75},
    'canakkale':      {'lat': 40.15, 'lon': 26.41, 'oran': 0.81},
    'denizli':        {'lat': 37.77, 'lon': 29.09, 'oran': 2.66},
    'diyarbakir':     {'lat': 37.91, 'lon': 40.23, 'oran': 4.71},
    'edirne':         {'lat': 41.68, 'lon': 26.56, 'oran': 1.15},
    'erzincan':       {'lat': 39.75, 'lon': 39.49, 'oran': 0.00},
    'erzurum':        {'lat': 39.90, 'lon': 41.27, 'oran': 0.00},
    'eskisehir':      {'lat': 39.78, 'lon': 30.52, 'oran': 1.79},
    'hakkari':        {'lat': 37.57, 'lon': 43.74, 'oran': 0.34},
    'isparta':        {'lat': 37.76, 'lon': 30.55, 'oran': 0.84},
    'istanbul':       {'lat': 41.01, 'lon': 28.97, 'oran': 24.43},
    'izmir':          {'lat': 38.42, 'lon': 27.14, 'oran': 11.51},
    'kars':           {'lat': 40.60, 'lon': 43.10, 'oran': 0.14},
    'kirklareli':     {'lat': 41.74, 'lon': 27.22, 'oran': 1.63},
    'kutahya':        {'lat': 39.42, 'lon': 29.98, 'oran': 1.01},
    'manisa':         {'lat': 38.61, 'lon': 27.43, 'oran': 3.36},
    'mardin':         {'lat': 37.31, 'lon': 40.74, 'oran': 2.16},
    'mugla':          {'lat': 37.21, 'lon': 28.36, 'oran': 3.99},
    'mus':            {'lat': 38.73, 'lon': 41.49, 'oran': 0.35},
    'siirt':          {'lat': 37.93, 'lon': 41.95, 'oran': 0.76},
    'sivas':          {'lat': 39.75, 'lon': 37.02, 'oran': 0.93},
    'tekirdag':       {'lat': 40.98, 'lon': 27.51, 'oran': 4.09},
    'tokat':          {'lat': 40.31, 'lon': 36.55, 'oran': 0.74},
    'sanliurfa':      {'lat': 37.16, 'lon': 38.79, 'oran': 5.48},
    'usak':           {'lat': 38.68, 'lon': 29.41, 'oran': 0.73},
    'van':            {'lat': 38.49, 'lon': 43.38, 'oran': 1.16},
    'yozgat':         {'lat': 39.82, 'lon': 34.80, 'oran': 0.62},
    'bayburt':        {'lat': 40.26, 'lon': 40.22, 'oran': 0.00},
    'batman':         {'lat': 37.88, 'lon': 41.13, 'oran': 1.74},
    'sirnak':         {'lat': 37.52, 'lon': 42.46, 'oran': 1.34},
    'ardahan':        {'lat': 41.11, 'lon': 42.70, 'oran': 0.01},
    'igdir':          {'lat': 39.92, 'lon': 44.04, 'oran': 0.00},
    'yalova':         {'lat': 40.65, 'lon': 29.27, 'oran': 0.57},
}

# Agirliklari normalize et (toplam = 1.0)
toplam_oran = sum(v['oran'] for v in SEHIRLER.values())
for k in SEHIRLER:
    SEHIRLER[k]['agirlik'] = SEHIRLER[k]['oran'] / toplam_oran if toplam_oran > 0 else 0

CFG = {
    'start_date':  '2016-01-01',
    'end_date':    '2026-03-01',

    # VMD
    'vmd_K': 8, 'vmd_alpha': 2000, 'vmd_tau': 0,
    'vmd_DC': 0, 'vmd_init': 1, 'vmd_tol': 1e-7,

    # PatchTST
    'patch_size': 6,
    'stride':     2,
    'seq_len':    168,
    'pred_len':   24,
    'd_model':    128,
    'n_heads':    8,
    'n_layers':   3,
    'dropout':    0.2,

    # Egitim
    'batch_size':     32,
    'epochs':         100,
    'lr':             1e-4,
    'train_ratio':    0.7,
    'val_ratio':      0.15,
    'patience':       15,
    'peak_threshold': 0.85,
    'peak_weight':    1.5,

    'weather_path': OUTPUT_DIR + 'weather_weighted.csv',
    'load_path':    EPIAS_PATH,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('v4 konfigurasyonu hazir. Cihaz: ' + str(DEVICE))
print('Toplam il: {}, Tarih: {} - {}'.format(len(SEHIRLER), CFG['start_date'], CFG['end_date']))

In [ ]:
# ============================================================
# HUCRE 5 -- AGIRLIKLI HAVA VERİSİ (41 İL)
# Open-Meteo: her il ayri cekilir, tuketim oraниyla agirliklandirilir
# ============================================================
def fetch_weather_city(sehir, lat, lon, start, end):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'hourly': ['temperature_2m', 'relativehumidity_2m', 'windspeed_10m',
                   'shortwave_radiation', 'direct_radiation'],
        'timezone': 'Europe/Istanbul',
        'wind_speed_unit': 'ms',
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    d = r.json()['hourly']
    df = pd.DataFrame({
        'tarih':      pd.to_datetime(d['time']),
        'sicaklik':   d['temperature_2m'],
        'nem':        d['relativehumidity_2m'],
        'ruzgar':     d['windspeed_10m'],
        'ghi':        d['shortwave_radiation'],
        'dni':        d['direct_radiation'],
    })
    return df

def build_weighted_weather():
    if os.path.exists(CFG['weather_path']):
        print('Mevcut agirlikli hava verisi yuklendi.')
        return pd.read_csv(CFG['weather_path'], parse_dates=['tarih'])

    print('41 il icin hava verisi cekiliyor (2010-2026)...')
    print('Bu islem ~10-15 dakika surebilir.')

    # Referans tarih dizisi
    tarihler = pd.date_range(CFG['start_date'], CFG['end_date'], freq='h')
    weighted = pd.DataFrame({'tarih': tarihler})
    weighted['w_sicaklik'] = 0.0
    weighted['w_nem']      = 0.0
    weighted['w_ruzgar']   = 0.0
    weighted['w_ghi']      = 0.0
    weighted['w_dni']      = 0.0

    basarili = 0
    for sehir, bilgi in tqdm(SEHIRLER.items(), desc='İller'):
        agirlik = bilgi['agirlik']
        if agirlik == 0:
            continue  # sifir agirlikli illeri atla
        try:
            df_s = fetch_weather_city(
                sehir, bilgi['lat'], bilgi['lon'],
                CFG['start_date'], CFG['end_date']
            )
            df_s = df_s.set_index('tarih').reindex(tarihler).ffill().bfill().reset_index()
            df_s.columns = ['tarih'] + list(df_s.columns[1:])

            weighted['w_sicaklik'] += agirlik * df_s['sicaklik'].values
            weighted['w_nem']      += agirlik * df_s['nem'].values
            weighted['w_ruzgar']   += agirlik * df_s['ruzgar'].values
            weighted['w_ghi']      += agirlik * df_s['ghi'].values
            weighted['w_dni']      += agirlik * df_s['dni'].values
            basarili += 1
            time.sleep(0.3)
        except Exception as e:
            print('HATA {}: {}'.format(sehir, e))

    weighted.to_csv(CFG['weather_path'], index=False)
    print('Agirlikli hava verisi kaydedildi. ({} il basarili)'.format(basarili))
    return weighted

weather_df = build_weighted_weather()
print('Hava verisi: {}'.format(weather_df.shape))
print(weather_df.head(3))

In [ ]:
# ============================================================
# HUCRE 6 -- EPİAŞ + TAKVİM FEATURES
# ============================================================
def load_epias(path):
    df = pd.read_csv(path, sep=';', decimal=',', encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # Tarih sutunu bul
    tarih_kol = [c for c in df.columns if 'arih' in c][0]
    saat_kol  = [c for c in df.columns if 'aat' in c][0]
    df['tarih'] = pd.to_datetime(
        df[tarih_kol].astype(str).str.strip() + ' ' + df[saat_kol].astype(str).str.strip(),
        dayfirst=True, errors='coerce'
    )

    # Tuketim sutunu bul
    kol = [c for c in df.columns if 'ketim' in c][0]
    df['gercek_tuketim_mwh'] = (
        df[kol].astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )
    df = df[['tarih', 'gercek_tuketim_mwh']].dropna().sort_values('tarih').reset_index(drop=True)
    print('EPİAS: {:,} satir | {} -> {}'.format(len(df), df.tarih.min(), df.tarih.max()))
    return df

def add_calendar_features(df):
    tr_hol    = holidays.Turkey(years=range(2016, 2027))
    tatil_set = set(tr_hol.keys())
    dt = pd.to_datetime(df['tarih'])

    df['saat']        = dt.dt.hour
    df['gun_haftada'] = dt.dt.dayofweek
    df['ay']          = dt.dt.month
    df['yil']         = dt.dt.year
    df['hafta_sonu']  = (dt.dt.dayofweek >= 5).astype(int)

    tarihler = dt.dt.date.tolist()
    df['resmi_tatil']   = [1 if d in tatil_set else 0 for d in tarihler]
    df['tatil_oncesi']  = [1 if (d + datetime.timedelta(days=1)) in tatil_set else 0 for d in tarihler]
    df['tatil_sonrasi'] = [1 if (d - datetime.timedelta(days=1)) in tatil_set else 0 for d in tarihler]

    # Ramazan
    ramazan_keys = [d for d, name in tr_hol.items() if 'Ramazan' in name or 'ramadan' in name.lower()]
    ramazan_set  = set(ramazan_keys)
    df['ramazan_flag'] = [1 if d in ramazan_set else 0 for d in tarihler]

    tat = df['resmi_tatil'].tolist()
    gun = df['gun_haftada'].tolist()
    ay  = df['ay'].tolist()
    df['okul_gunu']     = [int(gun[i]<5 and (ay[i]>=9 or ay[i]<=6) and tat[i]==0) for i in range(len(df))]
    df['tarife_dilimi'] = [0 if 6<=s<17 else (1 if 17<=s<22 else 2) for s in df['saat'].tolist()]
    df['pik_saati']     = ((df['saat']>=17) & (df['saat']<22) & (df['gun_haftada']<5)).astype(int)

    df['saat_sin'] = np.sin(2*np.pi*df['saat']/24)
    df['saat_cos'] = np.cos(2*np.pi*df['saat']/24)
    df['ay_sin']   = np.sin(2*np.pi*df['ay']/12)
    df['ay_cos']   = np.cos(2*np.pi*df['ay']/12)
    df['gun_sin']  = np.sin(2*np.pi*df['gun_haftada']/7)
    df['gun_cos']  = np.cos(2*np.pi*df['gun_haftada']/7)
    return df

# Master dataset olustur
if os.path.exists(CFG['load_path']):
    load_df = load_epias(CFG['load_path'])
else:
    print('UYARI: Demo veri kullaniliyor (2010-2026)')
    tarihler = pd.date_range(CFG['start_date'], CFG['end_date'], freq='h')
    yukler   = (35000 + 8000*np.sin(2*np.pi*np.arange(len(tarihler))/24)
                + 3000*np.sin(2*np.pi*np.arange(len(tarihler))/(24*365))
                + np.random.normal(0, 1000, len(tarihler)))
    load_df  = pd.DataFrame({'tarih': tarihler, 'gercek_tuketim_mwh': yukler})

df = pd.merge(load_df, weather_df, on='tarih', how='inner')
df = add_calendar_features(df)
df = df.ffill().bfill()
print('Master dataset v4: {:,} satir, {} sutun'.format(len(df), df.shape[1]))

In [ ]:
# ============================================================
# HUCRE 7 -- VMD (K=8)
# NOT: VMD tum veri uzerinde uygulanmistir.
# Akademik not: VMD parametrik bir model degil (frekans ayristirma),
# bu nedenle literaturde tum veri uzerinde uygulama kabul gorur.
# Methodology bolumunde belirtiniz.
# ============================================================
print('VMD uygulanıyor (K={}, 16 yillik veri)...'.format(CFG['vmd_K']))
signal = df['gercek_tuketim_mwh'].values
u, _, _ = VMD(signal, CFG['vmd_alpha'], CFG['vmd_tau'],
              CFG['vmd_K'], CFG['vmd_DC'], CFG['vmd_init'], CFG['vmd_tol'])

n = len(df)
for k in range(CFG['vmd_K']):
    imf = u[k]
    if len(imf) > n:   imf = imf[:n]
    elif len(imf) < n: imf = np.pad(imf, (0, n-len(imf)), mode='edge')
    df['imf_{}'.format(k+1)] = imf

print('VMD tamamlandi. {} IMF eklendi.'.format(CFG['vmd_K']))

fig, axes = plt.subplots(CFG['vmd_K'], 1, figsize=(14, 10))
for k in range(CFG['vmd_K']):
    axes[k].plot(df['imf_{}'.format(k+1)].values[:720], linewidth=0.8)
    axes[k].set_ylabel('IMF {}'.format(k+1), fontsize=8)
    axes[k].tick_params(labelsize=7)
plt.suptitle('VMD IMF (ilk 30 gun)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'imf_plot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# HUCRE 8 -- DATASET & DATALOADER (leakage yok)
# ============================================================
class LoadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def create_sequences(features, target, seq_len, pred_len):
    X, y = [], []
    for i in range(len(target) - seq_len - pred_len + 1):
        X.append(features[i:i+seq_len])
        y.append(target[i+seq_len:i+seq_len+pred_len])
    return np.array(X), np.array(y)

# Explicit feature list
imf_cols      = ['imf_{}'.format(k+1) for k in range(CFG['vmd_K'])]
weather_cols  = ['w_sicaklik', 'w_nem', 'w_ruzgar', 'w_ghi', 'w_dni']
calendar_cols = [
    'saat', 'gun_haftada', 'ay', 'yil', 'hafta_sonu',
    'resmi_tatil', 'tatil_oncesi', 'tatil_sonrasi',
    'okul_gunu', 'tarife_dilimi', 'pik_saati', 'ramazan_flag',
    'saat_sin', 'saat_cos', 'ay_sin', 'ay_cos', 'gun_sin', 'gun_cos',
]
feature_cols = imf_cols + weather_cols + calendar_cols
feature_cols = [c for c in feature_cols if c in df.columns]

print('Feature gruplari:')
print('  IMF:     {}'.format(len(imf_cols)))
print('  Hava:    {}'.format(len([c for c in weather_cols if c in df.columns])))
print('  Takvim:  {}'.format(len([c for c in calendar_cols if c in df.columns])))
print('  TOPLAM:  {}'.format(len(feature_cols)))

X_raw = df[feature_cols].values.astype(np.float32)
y_raw = df['gercek_tuketim_mwh'].values.astype(np.float32)

# Train/val/test split
n_total = len(X_raw)
n_train = int(n_total * CFG['train_ratio'])
n_val   = int(n_total * CFG['val_ratio'])

# Scaler SADECE train ile fit (leakage yok)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
scaler_X.fit(X_raw[:n_train])
scaler_y.fit(y_raw[:n_train].reshape(-1, 1))

X_sc = scaler_X.transform(X_raw)
y_sc = scaler_y.transform(y_raw.reshape(-1, 1)).flatten()

X, y     = create_sequences(X_sc, y_sc, CFG['seq_len'], CFG['pred_len'])
n        = len(X)
n_train2 = int(n * CFG['train_ratio'])
n_val2   = int(n * CFG['val_ratio'])

train_loader = DataLoader(LoadDataset(X[:n_train2],                y[:n_train2]),                batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(LoadDataset(X[n_train2:n_train2+n_val2], y[n_train2:n_train2+n_val2]), batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(LoadDataset(X[n_train2+n_val2:],         y[n_train2+n_val2:]),         batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

n_features = X.shape[2]
print('Train={:,} | Val={:,} | Test={:,}'.format(n_train2, n_val2, n-n_train2-n_val2))

In [ ]:
# ============================================================
# HUCRE 9 -- PEAK AWARE LOSS (global quantile)
# ============================================================
y_train_raw      = y_raw[:n_train]
GLOBAL_Q_HIGH    = float(np.quantile(y_train_raw, CFG['peak_threshold']))
GLOBAL_Q_LOW     = float(np.quantile(y_train_raw, 1 - CFG['peak_threshold']))
GLOBAL_Q_HIGH_SC = float(scaler_y.transform([[GLOBAL_Q_HIGH]])[0][0])
GLOBAL_Q_LOW_SC  = float(scaler_y.transform([[GLOBAL_Q_LOW]])[0][0])

print('Pik esigi:  {:.1f} MWh'.format(GLOBAL_Q_HIGH))
print('Vadi esigi: {:.1f} MWh'.format(GLOBAL_Q_LOW))

class PeakAwareLoss(nn.Module):
    def __init__(self, q_high, q_low, weight=1.5):
        super().__init__()
        self.q_high = q_high
        self.q_low  = q_low
        self.weight = weight
        self.huber  = nn.HuberLoss(reduction='none')
    def forward(self, pred, target):
        base        = self.huber(pred, target)
        peak_mask   = (target >= self.q_high).float()
        valley_mask = (target <= self.q_low).float()
        agirlik     = 1.0 + self.weight * (peak_mask + valley_mask)
        return (base * agirlik).mean()

print('PeakAwareLoss hazir.')

In [ ]:
# ============================================================
# HUCRE 10 -- PatchTST MODEL
# ============================================================
class PatchEmbedding(nn.Module):
    def __init__(self, seq_len, patch_size, stride, n_features, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (seq_len - patch_size) // stride + 1
        self.proj       = nn.Linear(patch_size * n_features, d_model)
        self.norm       = nn.LayerNorm(d_model)
    def forward(self, x):
        patches = [x[:, i*self.stride:i*self.stride+self.patch_size, :].reshape(x.size(0), -1)
                   for i in range(self.n_patches)]
        return self.norm(self.proj(torch.stack(patches, dim=1)))

class PatchTST(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, patch_size, stride,
                 d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.patch_embed = PatchEmbedding(seq_len, patch_size, stride, n_features, d_model)
        n_patches        = self.patch_embed.n_patches
        self.pos_embed   = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)
        self.dropout     = nn.Dropout(dropout)
        self.head_1h  = nn.Sequential(
            nn.Linear(n_patches*d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, 1)
        )
        self.head_24h = nn.Sequential(
            nn.Linear(n_patches*d_model, d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model, pred_len)
        )
    def forward(self, x, mode='24h'):
        x = self.dropout(self.patch_embed(x) + self.pos_embed)
        x = self.norm(self.transformer(x)).reshape(x.size(0), -1)
        return self.head_1h(x) if mode == '1h' else self.head_24h(x)
    def get_attention_weights(self, x):
        x = self.patch_embed(x) + self.pos_embed
        weights = []
        for layer in self.transformer.layers:
            _, w = layer.self_attn(x, x, x, need_weights=True, average_attn_weights=True)
            weights.append(w.detach().cpu())
            x = layer(x)
        return weights

model = PatchTST(
    seq_len=CFG['seq_len'], n_features=n_features, pred_len=CFG['pred_len'],
    patch_size=CFG['patch_size'], stride=CFG['stride'], d_model=CFG['d_model'],
    n_heads=CFG['n_heads'], n_layers=CFG['n_layers'], dropout=CFG['dropout']
).to(DEVICE)

print('Model parametresi: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('Patch sayisi: {}'.format(model.patch_embed.n_patches))

In [ ]:
# ============================================================
# HUCRE 11 -- EGİTİM
# ============================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
criterion = PeakAwareLoss(q_high=GLOBAL_Q_HIGH_SC, q_low=GLOBAL_Q_LOW_SC, weight=CFG['peak_weight'])

best_val     = float('inf')
patience_cnt = 0
tr_losses    = []
val_losses   = []

epoch_bar = tqdm(range(CFG['epochs']), desc='Egitim', unit='epoch')

for epoch in epoch_bar:
    model.train()
    tl = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = 0.7 * criterion(model(Xb,'24h'), yb) + 0.3 * criterion(model(Xb,'1h'), yb[:,0:1])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()

    model.eval()
    vl = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            vl += criterion(model(Xb,'24h'), yb).item()

    tl /= len(train_loader)
    vl /= len(val_loader)
    tr_losses.append(tl)
    val_losses.append(vl)
    scheduler.step()

    if vl < best_val:
        best_val     = vl
        patience_cnt = 0
        torch.save(model.state_dict(), OUTPUT_DIR + 'best_model.pt')
    else:
        patience_cnt += 1

    epoch_bar.set_postfix({
        'Train':    '{:.4f}'.format(tl),
        'Val':      '{:.4f}'.format(vl),
        'Best':     '{:.4f}'.format(best_val),
        'Patience': '{}/{}'.format(patience_cnt, CFG['patience']),
        'Ilerleme': '%{:.0f}'.format((epoch+1)/CFG['epochs']*100)
    })

    if patience_cnt >= CFG['patience']:
        print('Erken durdurma: {} epoch'.format(epoch+1))
        break

print('Egitim tamamlandi! En iyi Val Loss: {:.4f}'.format(best_val))

In [ ]:
# ============================================================
# HUCRE 12 -- DEĞERLENDİRME
# ============================================================
model.load_state_dict(torch.load(OUTPUT_DIR + 'best_model.pt', map_location=DEVICE, weights_only=True))
model.eval()

preds, actuals = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds.append(model(Xb.to(DEVICE), '24h').cpu().numpy())
        actuals.append(yb.numpy())

preds   = np.concatenate(preds,   axis=0)
actuals = np.concatenate(actuals, axis=0)
pi = scaler_y.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)
ai = scaler_y.inverse_transform(actuals.reshape(-1,1)).reshape(actuals.shape)

mae  = np.mean(np.abs(pi - ai))
rmse = np.sqrt(np.mean((pi - ai)**2))
mape = np.mean(np.abs((ai - pi) / (ai + 1e-8))) * 100

flat_ai  = ai.flatten()
flat_pi  = pi.flatten()
q85      = np.quantile(flat_ai, 0.85)
pik_mask = flat_ai >= q85
mape_pik = np.mean(np.abs((flat_ai[pik_mask] - flat_pi[pik_mask]) / (flat_ai[pik_mask] + 1e-8))) * 100

print('Genel  MAE:  {:,.2f} MWh'.format(mae))
print('Genel  RMSE: {:,.2f} MWh'.format(rmse))
print('Genel  MAPE: {:.2f}%'.format(mape))
print('Pik    MAPE: {:.2f}%  (ust %15)'.format(mape_pik))

In [ ]:
# ============================================================
# HUCRE 13 -- GÖRSELLEŞTİRME
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('PatchTST + VMD v4 -- Turkiye Elektrik Yuk Tahmini (41 il, 2010-2026)', fontsize=13)

axes[0,0].plot(tr_losses,  label='Train', color='steelblue')
axes[0,0].plot(val_losses, label='Val',   color='coral')
axes[0,0].set_title('Loss Egrisi'); axes[0,0].legend()

n = min(24, len(pi.flatten()))
axes[0,1].plot(ai.flatten()[:n], label='Gercek', color='steelblue', alpha=0.8)
axes[0,1].plot(pi.flatten()[:n], label='Tahmin', color='coral',     alpha=0.8)
axes[0,1].set_title('Tahmin vs Gercek (24 saat)'); axes[0,1].legend()

fp, fa  = pi.flatten(), ai.flatten()
q85     = np.quantile(fa, 0.85)
nm      = fa < q85
pm      = fa >= q85
axes[1,0].scatter(fa[nm], fp[nm], alpha=0.15, s=1, color='steelblue', label='Normal')
axes[1,0].scatter(fa[pm], fp[pm], alpha=0.4,  s=3, color='red',       label='Pik')
mn, mx = min(fa.min(),fp.min()), max(fa.max(),fp.max())
axes[1,0].plot([mn,mx],[mn,mx],'k--', linewidth=1)
axes[1,0].set_title('Scatter (kirmizi=pik)'); axes[1,0].legend()

axes[1,1].axis('off')
axes[1,1].text(0.05, 0.5,
    'TEST METRIKLERI v4\n\n'
    'Genel MAE:  {:,.1f} MWh\n'.format(mae) +
    'Genel RMSE: {:,.1f} MWh\n'.format(rmse) +
    'Genel MAPE: {:.2f}%\n'.format(mape) +
    'Pik   MAPE: {:.2f}%\n\n'.format(mape_pik) +
    '41 il agirlikli hava\n'
    '2010-2026 (16 yil)\n'
    'Patch={}h | IMF={}'.format(CFG['patch_size'], CFG['vmd_K']),
    transform=axes[1,1].transAxes, fontsize=11,
    verticalalignment='center', fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3)
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'sonuclar_v4.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# HUCRE 14 -- ATTENTION HARİTASI
# ============================================================
sample_X, _ = next(iter(test_loader))
model.eval()
with torch.no_grad():
    weights = model.get_attention_weights(sample_X[0].unsqueeze(0).to(DEVICE))

fig, axes = plt.subplots(1, len(weights), figsize=(6*len(weights), 5))
if len(weights) == 1: axes = [axes]
for i, (w, ax) in enumerate(zip(weights, axes)):
    sns.heatmap(w[0].numpy(), ax=ax, cmap='Blues')
    ax.set_title('Layer {} Attention'.format(i+1))
    ax.tick_params(labelsize=6)
plt.suptitle('Patch Attention Agirliklari v4', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'attention_map_v4.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# HUCRE 15 -- SONUCLAR
# ============================================================
print('Cikti dosyalari:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(OUTPUT_DIR + f) / 1024
    print('  {} ({:.1f} KB)'.format(f, size))
print('\nKaggle Output sekmesinden indirebilirsiniz.')